In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType
from pyspark.sql import functions as F
import os

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Patient")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Patient/"
silver_base_path = "../../data_lake/silver/silver_Patient/"

In [ ]:
#schema for patient resource


general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

communication_schema = ArrayType(StructType([
    StructField("language", StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ]), True),
    StructField("preferred", StringType(), True)
    
]))

#Marital schema

marital_schema = (StructType([
    StructField("coding", ArrayType(general_coding_schema), True),
    StructField("text", StringType(), True)
]))

#telecom schema

telecom_schema = ArrayType(StructType([
    StructField("system", StringType(), True),
    StructField("value", StringType(), True),
    StructField("use", StringType(), True)
]))

#marital status schema

marital_schema = StructType([
    StructField("coding", ArrayType(general_coding_schema), True),
    StructField("text", StringType(), True)
])

#language schema

language_schema = ArrayType(
    StructType([
        StructField("language",
            StructType([
                StructField("coding", ArrayType(general_coding_schema), True),
                StructField("text", StringType(), True)
        ]),True),
        StructField("preferred", StringType(), True) #FHIR data will have preferred key for language but synthea doesnt
    ])
)

# more will be added later


In [ ]:
#identifier schema
identifier_coding_schema = StructType([
    StructField("coding", ArrayType(general_coding_schema), True),
    StructField("text",   StringType(), True),
])


patient_identifier_type = ArrayType(
    StructType([
        StructField("type",   identifier_coding_schema, True),  
        StructField("system", StringType(), True),  
        StructField("value",  StringType(), True),  
    ])
)

def extract_patient_identifier(type_code:str):
    matches = F.filter(
        col("identifier"),
        lambda x: F.element_at(x["type"]["coding"], 1)["code"] == F.lit(type_code)
    )
    
    first_match = F.try_element_at(matches, F.lit(1))   # null if no match
    return first_match["value"]

In [ ]:
#name schema
patient_name_schema = ArrayType(
    StructType([
        StructField("use", StringType(), True),  
        StructField("family", StringType(), True),
        StructField("given", ArrayType(StringType()), True),
        StructField("prefix", ArrayType(StringType()), True),
        StructField("suffix", ArrayType(StringType()), True)
    ])
)

def extract_patient_name(name_part:str):
    if name_part == "family":
        return col("patient_name")[0]['family']
    elif name_part == "first_name":
        return col("patient_name")[0]['given'][0]
    elif name_part == "middle_name":
        return F.when(
            F.size(col("patient_name")[0]["given"]) > 1,
            col("patient_name")[0]["given"][1]
        ).otherwise(F.lit(None).cast("string"))
    elif name_part == "prefix":
        return col("patient_name")[0]['prefix'][0]
    elif name_part == "suffix":
        return col("patient_name")[0]['suffix'][0]
    else:
        return None

In [ ]:
#address schema
address_extension_schema = ArrayType(StructType([
    # StructField("url", StringType(), True), i dont need it now 
    StructField("extension", ArrayType(
        StructType([
            StructField("url",StringType(),True),
            StructField("valueDecimal",DoubleType(),True)
        ])
    ), True)
])



)
address_schema = StructType([
        StructField("extension",address_extension_schema, True),
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])


In [ ]:
#extension schema which contains few details of patients - race, ethnicity, etc
extension_schema = ArrayType(StructType([
    StructField("url", StringType(), True),
    StructField("extension", ArrayType(
        StructType([
            StructField("url",StringType(),True),
            StructField("valueCoding",general_coding_schema ,True)
        ])
    ), True),
    StructField("valueString", StringType(), True),
    StructField("valueCode", StringType(), True),
    StructField("valueAddress", address_schema, True),  # reusing the address schema because city, state and country keys are common
]))

def extract_extension_value(extension_url:str):
    filtered_value = F.filter(col("extension"), lambda x: x["url"].contains(extension_url))[0]
    # if not filtered_value:
    #     return None
    if extension_url == "us-core-race" or extension_url == "us-core-ethnicity":
        return filtered_value["extension"][0]["valueCoding"]["display"]
    elif extension_url == "patient-mothersMaidenName":
        return filtered_value["valueString"]
    elif extension_url == "us-core-birthsex":
        return filtered_value["valueCode"]
    elif extension_url == "patient-birthPlace":
        return [filtered_value["valueAddress"]["city"] , filtered_value["valueAddress"]["state"], filtered_value["valueAddress"]["country"]]
    else:
        return None

In [ ]:
df_patient = spark.read.format("parquet").load(bronze_path)
df_patient.printSchema()

In [ ]:
df_intermediate_1 = df_patient.select(
    F.conv(F.substring(F.md5("resource.id"), 1, 15), 16, 10).cast("bigint").alias("patient_key"),
    col("resource.id").alias("patient_id"),
    from_json(col("resource.extension"), extension_schema).alias("extension"),
    from_json(col("resource.identifier"), patient_identifier_type).alias("identifier"),
    from_json(col("resource.name"), patient_name_schema).alias("patient_name"),
    from_json(col("resource.telecom"),telecom_schema)[0]["value"].alias("phone_number"),
    col("resource.gender").alias("gender"),
    col("resource.birthDate").alias("birthDate"),
    col("resource.deceasedDateTime").alias("deceasedDateTime"),
    from_json(col("resource.address"), ArrayType(address_schema)).alias("patient_address"),
    from_json(col("resource.maritalStatus"), marital_schema).alias("maritalStatus"),
    col("resource.multipleBirthBoolean").cast("boolean").alias("is_multiple_birth"),
    from_json(col("resource.communication"), language_schema).alias("communication"),
    col("input_file_name")
).drop("resource","ingestion_timestamp")

In [ ]:
df_patient_flattened = (df_intermediate_1
            .select(
                col("patient_key"),
                col("patient_id"),
                extract_patient_identifier("MR").alias("medical_record_number"),
                extract_patient_identifier("SS").alias("social_security_number"),
                extract_patient_identifier("DL").alias("drivers_license_number"),
                extract_patient_identifier("PPN").alias("passport_number"),
                extract_patient_name("first_name").alias("first_name"),
                extract_patient_name("middle_name").alias("middle_name"),
                extract_patient_name("family").alias("family_name"),
                extract_patient_name("prefix").alias("prefix"),
                # extract_patient_name("suffix").alias("suffix"),  # if needed we can include later
                extract_extension_value("us-core-race").alias("race"),
                extract_extension_value("us-core-ethnicity").alias("ethnicity"),
                extract_extension_value("patient-mothersMaidenName").alias("mothers_maiden_name"),
                extract_extension_value("us-core-birthsex").alias("birth_sex"),
                extract_extension_value("patient-birthPlace")[0].alias("birth_city"),
                extract_extension_value("patient-birthPlace")[1].alias("birth_state"),
                extract_extension_value("patient-birthPlace")[2].alias("birth_country"),
                col("gender"),
                col("birthDate").cast("date").alias("birth_date"),
                col("deceasedDateTime").cast("timestamp").alias("deceased_date_time"),
                col("maritalStatus")["coding"][0]["code"].alias("marital_status_code"),
                col("maritalStatus")["coding"][0]["display"].alias("marital_status_display"),
                F.array_join(col("patient_address")[0]["line"], ", ").alias("address_line"),
                col("patient_address")[0]['city'].alias("city"),
                col("patient_address")[0]['state'].alias("state"),
                col("patient_address")[0]['postalCode'].alias("postal_code"),
                col("patient_address")[0]['country'].alias("country"),
                col("patient_address")[0]['extension'][0]['extension'][0]['valueDecimal'].alias("latitude"),
                col("patient_address")[0]['extension'][0]['extension'][1]['valueDecimal'].alias("longitude"),
                col("phone_number"),
                col("communication")[0]["language"]["coding"][0]["code"].alias("language_code"),
                col("communication")[0]["language"]["coding"][0]["display"].alias("language_display"),
                col("is_multiple_birth"),
                col("input_file_name")
            ).withColumn("silver_timestamp",current_timestamp())
)

In [ ]:
df_patient_flattened.write.mode("overwrite").option("overwriteSchema", "true").parquet(silver_base_path)

In [ ]:
spark.stop()